# Features, targets and leakage

Cheat sheet for building time-series features **that would actually have been available at prediction time**.
Every section: what the operation does, how to write it, and how it leaks if you get it wrong.

What's in here
- Writing the prediction problem down before touching code
- Target construction with `shift(-h)` and the dropna alignment trap
- Lag, rolling, diff and pct_change features (and the `shift(1)` before `rolling`)
- Calendar features: numeric vs one-hot vs sin/cos
- Exogenous variables: actual weather vs the forecast that existed at decision time (`merge_asof`)
- Scaler leakage, target-encoding leakage, overlapping labels, bfill/interpolate leakage
- `resample` label/closed semantics
- A 12-question leakage checklist

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
df = df.sort_values("time").reset_index(drop=True)
print(df.shape, df["time"].dt.tz)
df.head(3)

(17520, 6) UTC


,time,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
0,2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83
1,2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21
2,2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71


## 0. Write the problem down first

> **At time t, using only information available at t, predict y at t + h.**

Three things must be fixed before any feature is written:
1. **Decision time** t (when the forecast is made / the trade is placed).
2. **Horizon** h (target time t + h).
3. **Information set** at t: which columns, at which timestamps, were known *at t*.

**Interview check:** "What exactly is one row, and at what moment would you have had these numbers?" If you cannot answer, you cannot judge leakage.

In this notebook a row is one hour of GB-style system data: consumption (MWh), temperature, wind, solar and day-ahead price.

## 1. Target construction: `shift(-h)`

`y.shift(-h)` moves future values *up* so the target for row t is `y[t+h]`. The last h rows become NaN (no future yet).

In [2]:
y = df["consumption_mwh"]
tgt = pd.DataFrame({
    "time": df["time"],
    "y_now": y,
    "y_h1": y.shift(-1),     # value one hour later
    "y_h24": y.shift(-24),   # value one day later
})
print(tgt.head(3), "\n")
print(tgt.tail(3), "\n")
# sanity: the h=24 target for row 0 must equal the actual value 24 rows later
assert tgt.loc[0, "y_h24"] == tgt.loc[24, "y_now"]
print("NaNs at the end:", tgt["y_h1"].isna().sum(), tgt["y_h24"].isna().sum())

                       time    y_now     y_h1    y_h24
0 2022-01-01 00:00:00+00:00  26858.4  26177.8  26785.3
1 2022-01-01 01:00:00+00:00  26177.8  26229.4  25826.5
2 2022-01-01 02:00:00+00:00  26229.4  25381.3  25799.8 

                           time    y_now     y_h1  y_h24
17517 2023-12-31 21:00:00+00:00  30263.2  28415.8    NaN
17518 2023-12-31 22:00:00+00:00  28415.8  27054.8    NaN
17519 2023-12-31 23:00:00+00:00  27054.8      NaN    NaN 

NaNs at the end: 1 24


**Pitfall: `dropna` on X and y separately.** They lose *different* rows (features lose the start, target loses the end), so `model.fit(X, y)` either fails on length or, worse, silently mis-aligns if you converted to numpy first.

In [3]:
X = pd.DataFrame({"lag1": y.shift(1), "lag24": y.shift(24)})
y24 = y.shift(-24)

X_bad = X.dropna()
y_bad = y24.dropna()
print("len X:", len(X_bad), " len y:", len(y_bad))
print("indices identical?", X_bad.index.equals(y_bad.index))
print("first X row:", X_bad.index[0], " first y row:", y_bad.index[0])
print("last  X row:", X_bad.index[-1], " last  y row:", y_bad.index[-1])

# .values would hide the mismatch until fit() complains about length ... or doesn't
try:
    LinearRegression().fit(X_bad.values, y_bad.values)
except ValueError as e:
    print("\nsklearn:", type(e).__name__, "-", str(e)[:80])

len X: 17496  len y: 17496
indices identical? False
first X row: 24  first y row: 0
last  X row: 17519  last  y row: 17495


**Correct pattern:** put target and features in *one* frame, `dropna` once, then split into X and y. Alignment is then guaranteed by construction.

In [4]:
frame = X.assign(y=y24).dropna()
X_ok, y_ok = frame.drop(columns="y"), frame["y"]
print(len(X_ok), len(y_ok), X_ok.index.equals(y_ok.index))
frame.head(3)

17472 17472 True


,lag1,lag24,y
24,27884.6,26858.4,28397.1
25,26785.3,26177.8,27134.3
26,25826.5,26229.4,26810.0


## 2. Lag features: `shift(k)`

`shift(k)` with positive k looks *back*: row t gets the value from t−k. Common lags for hourly data: 1 (last hour), 24 (same hour yesterday), 168 (same hour last week).

**Pitfall:** `shift` is positional, not time-based. If hours are missing or the frame is unsorted, `shift(24)` is *not* "24 hours ago". Check `time.diff().value_counts()` first.

In [5]:
print("sampling interval:", df["time"].diff().value_counts().head(3).to_dict())

lags = pd.DataFrame({
    "time": df["time"],
    "y": y,
    "lag1": y.shift(1),
    "lag24": y.shift(24),
    "lag168": y.shift(168),
})
# proof that lag24 at row t really is the value 24 hours earlier
i = 500
print(lags.loc[i, ["time", "lag24"]].to_dict(), "\nvalue 24 rows earlier:", lags.loc[i - 24, "y"],
      "at", lags.loc[i - 24, "time"])
lags.iloc[166:170]

sampling interval: {Timedelta('0 days 01:00:00'): 17519}
{'time': Timestamp('2022-01-21 20:00:00+0000', tz='UTC'), 'lag24': 37194.5} 
value 24 rows earlier: 37194.5 at 2022-01-20 20:00:00+00:00


,time,y,lag1,lag24,lag168
166,2022-01-07 22:00:00+00:00,33261.3,34361.0,31521.7,NaN
167,2022-01-07 23:00:00+00:00,31353.0,33261.3,30302.8,NaN
168,2022-01-08 00:00:00+00:00,28210.5,31353.0,28476.8,26858.4
169,2022-01-08 01:00:00+00:00,26205.3,28210.5,28227.5,26177.8


## 3. Rolling features and the `shift(1)` before `rolling`

`y.rolling(24).mean()` at row t averages `y[t-23 .. t]` — it **includes the current row**. Whether that leaks depends on the problem:

- Target is `y[t]` itself (nowcast / explaining consumption at t) → the feature contains 1/24 of the target → **leak**.
- Target is `y[t+h]` and `y[t]` is known at t → including `y[t]` is fine.

The safe habit: build every feature as of the decision time explicitly, `y.shift(1).rolling(24).mean()` = mean of `y[t-24 .. t-1]`.

In [6]:
roll = pd.DataFrame({
    "y": y,
    "roll24_incl_now": y.rolling(24).mean(),            # includes y[t]
    "roll24_shifted":  y.shift(1).rolling(24).mean(),   # y[t-24..t-1]
    "roll24_centered": y.rolling(24, center=True).mean(),  # uses 12 FUTURE hours -> never a feature
})
print("correlation with y[t]:")
print(roll.corr()["y"].round(4))
roll.iloc[23:26]

correlation with y[t]:
y                  1.0000
roll24_incl_now    0.4954
roll24_shifted     0.4885
roll24_centered    0.5286
Name: y, dtype: float64


,y,roll24_incl_now,roll24_shifted,roll24_centered
23,27884.6,30587.241667,NaN,30316.487500
24,26785.3,30584.195833,30587.241667,30264.195833
25,25826.5,30569.558333,30584.195833,30250.608333


**Quantify the leak.** Nowcast `y[t]` from calendar + one rolling feature, chronological 80/20 split. The un-shifted window contains `y[t]` with weight 1/window, so the shorter the window the bigger the fake improvement.

In [7]:
cal = pd.DataFrame({"hour": df["time"].dt.hour, "dow": df["time"].dt.dayofweek})
split = int(len(df) * 0.8)

def fit_report(feature_col, feats):
    fr = pd.concat([cal, feats, y.rename("y")], axis=1).dropna()
    tr, te = fr.iloc[:split], fr.iloc[split:]
    cols = ["hour", "dow", feature_col]
    m = LinearRegression().fit(tr[cols], tr["y"])
    r2 = r2_score(te["y"], m.predict(te[cols]))
    coef = dict(zip(cols, m.coef_.round(3)))
    return r2, coef

for w in (3, 24):
    feats = pd.DataFrame({f"roll{w}_incl_now": y.rolling(w).mean(),
                          f"roll{w}_shifted": y.shift(1).rolling(w).mean()})
    for col in feats.columns:
        r2, coef = fit_report(col, feats[[col]])
        print(f"{col:17s}  test R2 = {r2:.4f}   coef on feature = {coef[col]}")

roll3_incl_now     test R2 = 0.8818   coef on feature = 1.053
roll3_shifted      test R2 = 0.5893   coef on feature = 0.839
roll24_incl_now    test R2 = 0.4551   coef on feature = 0.929
roll24_shifted     test R2 = 0.4508   coef on feature = 0.917


The leaked version gets a higher R² than the honest one built from the same information; the honest version is what you could actually run. With `rolling(24)` the leak is only 1/24 of the target and easy to miss — that is exactly why it survives code review.

**Interview check:** "Would `shift(1)` be necessary here?" → Answer with the decision time, not with a rule of thumb.

## 4. `diff` and `pct_change`

`diff(k)` = `y[t] - y[t-k]`; `pct_change(k)` = `y[t]/y[t-k] - 1`. Both look back, both are fine as features at t. `pct_change` explodes near zero (prices!) and is undefined across sign changes.

In [8]:
p = df["price_eur_mwh"]
chg = pd.DataFrame({
    "price": p,
    "diff1": p.diff(1),
    "diff24": p.diff(24),
    "pct1": p.pct_change(1),
})
print("largest |pct_change| — all near-zero or negative prices:")
print(chg.reindex(chg["pct1"].abs().sort_values(ascending=False).index).head(4).round(3))
print("\nnegative prices in sample:", (p < 0).sum(), "| within 5 EUR of zero:", (p.abs() < 5).sum())

largest |pct_change| — all near-zero or negative prices:
        price   diff1  diff24     pct1
10449  116.83  117.26   18.80 -272.698
12863   78.83   79.67   41.01  -94.845
15913   58.43   59.19    4.84  -77.882
16008   99.68  101.24   50.77  -64.897

negative prices in sample: 43 | within 5 EUR of zero: 27


**Pitfall:** `pct_change()` by default fills NaNs forward before computing in older pandas (`fill_method="pad"`); in pandas ≥ 2.1 this is deprecated. Pass `fill_method=None` explicitly so a gap gives NaN instead of a fake 0% change.

In [9]:
s = pd.Series([100.0, np.nan, np.nan, 110.0])
print("fill_method=None :", s.pct_change(fill_method=None).round(3).tolist())

fill_method=None : [nan, nan, nan, nan]


## 5. Calendar features: numeric, one-hot, cyclic

`hour` as a number says 23 is far from 0 — wrong for a linear model. One-hot fixes it (24 columns). sin/cos gives 2 columns and keeps adjacency. Use `pd.get_dummies(..., dtype=float)` and make sure train and test end up with the *same* columns.

In [10]:
t = df["time"]
calendar = pd.DataFrame({
    "hour": t.dt.hour, "dow": t.dt.dayofweek, "month": t.dt.month,
    "is_weekend": (t.dt.dayofweek >= 5).astype(int),
})
calendar["hour_sin"] = np.sin(2 * np.pi * calendar["hour"] / 24)
calendar["hour_cos"] = np.cos(2 * np.pi * calendar["hour"] / 24)
calendar["dow_sin"] = np.sin(2 * np.pi * calendar["dow"] / 7)
calendar["dow_cos"] = np.cos(2 * np.pi * calendar["dow"] / 7)

hour_1h = pd.get_dummies(calendar["hour"], prefix="h", dtype=float)   # 24 columns
print(hour_1h.shape)
calendar.iloc[[0, 6, 12, 23]]

(17520, 24)


,hour,dow,month,is_weekend,hour_sin,hour_cos,dow_sin,dow_cos
0,0,5,1,1,0.000000e+00,1.000000e+00,-0.974928,-0.222521
6,6,5,1,1,1.000000e+00,6.123234e-17,-0.974928,-0.222521
12,12,5,1,1,1.224647e-16,-1.000000e+00,-0.974928,-0.222521
23,23,5,1,1,-2.588190e-01,9.659258e-01,-0.974928,-0.222521


In [11]:
# same target (h=24), same lags, three encodings of hour
base = pd.DataFrame({"lag24": y.shift(24), "lag48": y.shift(48)})
encodings = {
    "numeric hour": calendar[["hour", "is_weekend"]],
    "sin/cos hour": calendar[["hour_sin", "hour_cos", "is_weekend"]],
    "one-hot hour": pd.concat([hour_1h, calendar[["is_weekend"]]], axis=1),
}
for name, enc in encodings.items():
    fr = pd.concat([base, enc, y.shift(-24).rename("y")], axis=1).dropna()
    tr, te = fr.iloc[:split], fr.iloc[split:]
    Xc = [c for c in fr.columns if c != "y"]
    m = LinearRegression().fit(tr[Xc], tr["y"])
    print(f"{name:13s} test R2 = {r2_score(te['y'], m.predict(te[Xc])):.4f}")

numeric hour  test R2 = 0.8040
sin/cos hour  test R2 = 0.8084


one-hot hour  test R2 = 0.8273


## 6. Exogenous variables: what did you know at decision time?

Predicting consumption at **t+24** using **temperature at t+24** is only legitimate if that number existed at t. The actual reading did not. What existed was a **weather forecast** issued at some `origin_datetime ≤ t` for `forecast_datetime = t+24`.

**Interview check:** "When would this variable actually have become available?"

The forecast file: issued every 00:00 and 12:00 UTC, horizons 1–48h, so each target hour has up to four forecasts of different ages.

In [12]:
fc = pd.read_csv("../data/weather_forecasts.csv", parse_dates=["origin_datetime", "forecast_datetime"])
print(fc.shape)
one_hour = fc[fc["forecast_datetime"] == pd.Timestamp("2022-03-10 15:00", tz="UTC")]
one_hour

(70004, 4)


,origin_datetime,forecast_datetime,horizon_h,temp_forecast_c
6470,2022-03-09 00:00:00+00:00,2022-03-10 15:00:00+00:00,39,6.51
6506,2022-03-09 12:00:00+00:00,2022-03-10 15:00:00+00:00,27,7.53
6542,2022-03-10 00:00:00+00:00,2022-03-10 15:00:00+00:00,15,7.99
6578,2022-03-10 12:00:00+00:00,2022-03-10 15:00:00+00:00,3,6.00


Build the frame in **decision-time coordinates**: each row is a decision at t, target time t+24. Then attach the temperature three ways.

In [13]:
h = 24
frame = pd.DataFrame({
    "decision_time": df["time"],
    "target_time":   df["time"] + pd.Timedelta(hours=h),
    "y":             y.shift(-h),                 # consumption at t+24
    "cons_now":      y,                           # known at t
    "cons_lag24":    y.shift(24),                 # known at t
    "cons_lag144":   y.shift(144),                # same hour last week relative to the TARGET
    "temp_now":      df["temp_c"],                # known at t   (very honest)
    "temp_actual_target": df["temp_c"].shift(-h), # measured at t+24 (NOT known at t -> cheating)
})
tt = frame["target_time"]
frame["hour_sin"] = np.sin(2 * np.pi * tt.dt.hour / 24)
frame["hour_cos"] = np.cos(2 * np.pi * tt.dt.hour / 24)
frame["is_weekend"] = (tt.dt.dayofweek >= 5).astype(int)
frame.head(2)

,decision_time,target_time,y,cons_now,cons_lag24,cons_lag144,temp_now,temp_actual_target,hour_sin,hour_cos,is_weekend
0,2022-01-01 00:00:00+00:00,2022-01-02 00:00:00+00:00,26785.3,26858.4,NaN,NaN,0.11,0.15,0.000000,1.000000,1
1,2022-01-01 01:00:00+00:00,2022-01-02 01:00:00+00:00,25826.5,26177.8,NaN,NaN,-0.18,-0.59,0.258819,0.965926,1


**Wrong join:** for each target hour take the *most recent* forecast (smallest horizon). That forecast was issued *after* the decision time.

In [14]:
fc_r = fc.rename(columns={"forecast_datetime": "target_time"})
latest_per_target = (fc_r.sort_values("origin_datetime")
                         .groupby("target_time", as_index=False).tail(1))
wrong = frame.merge(latest_per_target, on="target_time", how="left")
print("horizon of the forecast used:", wrong["horizon_h"].describe()[["min", "max"]].to_dict())
print("share of rows whose forecast was issued AFTER the decision:",
      (wrong["origin_datetime"] > wrong["decision_time"]).mean().round(3))

horizon of the forecast used: {'min': 1.0, 'max': 12.0}
share of rows whose forecast was issued AFTER the decision: 0.999


**Right join:** `pd.merge_asof` — for each row take the forecast for *that* target hour (`by=`) with the latest `origin_datetime ≤ decision_time` (`direction="backward"`). Both sides must be sorted by the `on` key.

In [15]:
honest = pd.merge_asof(
    frame.sort_values("decision_time"),
    fc_r.sort_values("origin_datetime")[["origin_datetime", "target_time", "horizon_h", "temp_forecast_c"]],
    left_on="decision_time", right_on="origin_datetime",
    by="target_time", direction="backward",
)
print("horizon of the forecast used:", honest["horizon_h"].describe()[["min", "max"]].to_dict())
print("any forecast issued after decision?", (honest["origin_datetime"] > honest["decision_time"]).any())
print("rows without a forecast:", honest["temp_forecast_c"].isna().sum())
honest[["decision_time", "target_time", "origin_datetime", "horizon_h", "temp_forecast_c", "temp_actual_target"]].iloc[[100, 106, 112]]

horizon of the forecast used: {'min': 24.0, 'max': 35.0}
any forecast issued after decision? False
rows without a forecast: 24


,decision_time,target_time,origin_datetime,horizon_h,temp_forecast_c,temp_actual_target
100,2022-01-05 04:00:00+00:00,2022-01-06 04:00:00+00:00,2022-01-05 00:00:00+00:00,28.0,-0.84,-2.85
106,2022-01-05 10:00:00+00:00,2022-01-06 10:00:00+00:00,2022-01-05 00:00:00+00:00,34.0,1.16,0.99
112,2022-01-05 16:00:00+00:00,2022-01-06 16:00:00+00:00,2022-01-05 12:00:00+00:00,28.0,5.72,1.25


In [16]:
# always sanity-check the join: forecast error should be a few degrees, biased slightly warm, worse at longer horizons
e = honest["temp_forecast_c"] - honest["temp_actual_target"]
print("forecast error mean %.2f  sd %.2f" % (e.mean(), e.std()))
print(e.groupby(honest["horizon_h"] // 4 * 4).std().round(2).rename("sd_by_horizon_bucket"))

forecast error mean 0.30  sd 2.19
horizon_h
24.0    1.92
28.0    2.18
32.0    2.42
Name: sd_by_horizon_bucket, dtype: float64


Now the comparison that matters: **cheating** (actual temperature at target) vs **honest** (forecast available at t) vs **very honest** (last observed temperature).

In [17]:
common = ["cons_now", "cons_lag24", "cons_lag144", "hour_sin", "hour_cos", "is_weekend"]
variants = {
    "actual temp at t+24 (cheat)": ["temp_actual_target"],
    "forecast temp known at t   ": ["temp_forecast_c"],
    "temp at t (lagged)         ": ["temp_now"],
    "no temperature             ": [],
}
fr = honest.dropna(subset=common + ["y", "temp_actual_target", "temp_forecast_c", "temp_now"])
tr, te = fr.iloc[:int(len(fr) * .8)], fr.iloc[int(len(fr) * .8):]
for name, extra in variants.items():
    cols = common + extra
    m = LinearRegression().fit(tr[cols], tr["y"])
    pred = m.predict(te[cols])
    print(f"{name}  R2 = {r2_score(te['y'], pred):.4f}   RMSE = {mean_squared_error(te['y'], pred) ** .5:8.1f}")

actual temp at t+24 (cheat)  R2 = 0.9169   RMSE =   1131.2
forecast temp known at t     R2 = 0.9145   RMSE =   1147.2
temp at t (lagged)           R2 = 0.9089   RMSE =   1184.1
no temperature               R2 = 0.9074   RMSE =   1193.9


The "cheat" number is the one you would report by accident if you merged actuals on `time`. It is not achievable in production. The gap between cheat and honest is the *value of a perfect weather forecast*, not model skill.

## 7. Scaler leakage

`StandardScaler().fit(X)` on the full sample uses test-period means/variances. With a trend or regime change the test mean differs from the train mean, and the scaled test features carry that information. Fit on train only, or use a `Pipeline` so it cannot happen.

In [18]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

Xf = fr[["cons_now", "temp_forecast_c"]]
sc_full = StandardScaler().fit(Xf)
sc_train = StandardScaler().fit(Xf.iloc[:int(len(Xf) * .8)])
print(pd.DataFrame({"mean_full": sc_full.mean_, "mean_train": sc_train.mean_,
                    "sd_full": sc_full.scale_, "sd_train": sc_train.scale_}, index=Xf.columns).round(2))

# the safe way: scaler lives inside the pipeline and is fit only on whatever .fit() sees
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(tr[common], tr["y"])
print("\npipeline test R2:", round(r2_score(te["y"], pipe.predict(te[common])), 4))

                 mean_full  mean_train  sd_full  sd_train
cons_now          29290.53    29354.17  4202.80   4265.85
temp_forecast_c      10.31       10.31     7.02      7.30

pipeline test R2: 0.9074


## 8. Group means / target encoding computed on the full sample

"Mean consumption for this hour-of-day" is a fine feature — if the mean is computed on **train only**. Computed on the full sample, every test row's feature contains its own target. Flagrant version: the mean of the target over the *same day* — it contains this row's target and the whole day's level, none of which exists at decision time.

In [19]:
day_key = fr["target_time"].dt.floor("D")
enc_leak = fr["y"].groupby(day_key).transform("mean")   # mean of the 24 targets of that day, incl. this row's own target

hd = tr.groupby([tr["target_time"].dt.hour, tr["target_time"].dt.dayofweek])["y"].mean()   # train only
hd.index.names = ["hour", "dow"]
enc_ok = pd.Series(pd.MultiIndex.from_arrays([fr["target_time"].dt.hour, fr["target_time"].dt.dayofweek]).map(hd),
                   index=fr.index)

X_ok = pd.DataFrame({"hd_train": enc_ok, "cons_lag24": fr["cons_lag24"]})
X_leak = X_ok.assign(daily_mean_full=enc_leak)
for name, Xe in [("honest: hour x dow mean from train + lag24", X_ok),
                 ("  + daily mean of y from full sample (leak)", X_leak)]:
    m = LinearRegression().fit(Xe.loc[tr.index], tr["y"])
    print(f"{name}  test R2 = {r2_score(te['y'], m.predict(Xe.loc[te.index])):.4f}")

honest: hour x dow mean from train + lag24  test R2 = 0.8655
  + daily mean of y from full sample (leak)  test R2 = 0.9421


Any statistic computed with `groupby(...).transform(...)` on the whole frame before the split is suspect: `transform("mean")`, `transform("std")`, `rank(pct=True)`, z-scores by group. Compute on train, then `map`/`merge` onto test.

## 9. Overlapping / autocorrelated labels and shuffled CV

With h = 24 the targets of rows t and t+1 are consumption one hour apart — nearly the same number. Shuffled K-fold puts row t in test and rows t±1 in train, so a flexible model "predicts" test by copying neighbours. `TimeSeriesSplit` (or a manual chronological split with a gap) does not.

In [20]:
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score
from sklearn.tree import DecisionTreeRegressor

y24_target = fr["y"]
print("autocorrelation of the h=24 target at lag 1/2/24:",
      [round(y24_target.autocorr(k), 3) for k in (1, 2, 24)])

Xn = fr[["cons_now", "cons_lag24", "hour_sin", "hour_cos"]]
tree = DecisionTreeRegressor(random_state=0)             # deliberately over-flexible
for name, cv in [("KFold shuffle=True", KFold(5, shuffle=True, random_state=0)),
                 ("TimeSeriesSplit   ", TimeSeriesSplit(5))]:
    sc = cross_val_score(tree, Xn, y24_target, cv=cv, scoring="r2")
    print(f"{name}  mean R2 = {sc.mean():.3f}   folds = {np.round(sc, 3)}")

autocorrelation of the h=24 target at lag 1/2/24: [0.935, 0.781, 0.914]


KFold shuffle=True  mean R2 = 0.715   folds = [0.697 0.724 0.716 0.716 0.72 ]
TimeSeriesSplit     mean R2 = 0.636   folds = [0.54  0.636 0.628 0.723 0.655]


**Interview check:** "Why shouldn't we shuffle?" → autocorrelated labels + neighbour rows in train. Also: shuffling estimates *interpolation* skill; the job is *extrapolation* in time.

Add a **gap** of at least h rows between train end and test start when h > 1, otherwise the last train targets are literally future values relative to the first test decision time (`TimeSeriesSplit(gap=24)`).

## 10. Filling gaps: `ffill` is stale, `bfill` and `interpolate` are from the future

`bfill` copies the *next* observed value backwards. Linear `interpolate` uses the next value too. Both are fine for cleaning a *feature you receive with a lag*, but as features "as of t" they leak.

In [21]:
s = pd.Series([10.0, np.nan, np.nan, np.nan, 20.0], index=pd.date_range("2023-01-01", periods=5, freq="h", tz="UTC"))
pd.DataFrame({"raw": s, "ffill": s.ffill(), "bfill (future)": s.bfill(),
              "interpolate (future)": s.interpolate(), "ffill limit=1": s.ffill(limit=1)})

,raw,ffill,bfill (future),interpolate (future),ffill limit=1
2023-01-01 00:00:00+00:00,10.0,10.0,10.0,10.0,10.0
2023-01-01 01:00:00+00:00,NaN,10.0,20.0,12.5,10.0
2023-01-01 02:00:00+00:00,NaN,10.0,20.0,15.0,NaN
2023-01-01 03:00:00+00:00,NaN,10.0,20.0,17.5,NaN
2023-01-01 04:00:00+00:00,20.0,20.0,20.0,20.0,20.0


## 11. `resample` semantics: `label` and `closed`

Daily mean via `resample("D").mean()` is labelled at **00:00 of that day** but contains data up to 23:00 — it is known only at the *next* midnight. Shift the availability time before joining it back to hourly rows.

In [22]:
hourly = df.set_index("time")["consumption_mwh"]
daily = hourly.resample("D").mean()
print("label 2022-01-05 covers:", hourly["2022-01-05"].index.min(), "->", hourly["2022-01-05"].index.max())
print("daily mean labelled 2022-01-05:", round(daily.loc[pd.Timestamp("2022-01-05", tz="UTC")], 1))

# make the availability time explicit: this number exists from the following midnight
daily_avail = daily.rename("prev_day_mean").reset_index()
daily_avail["available_from"] = daily_avail["time"] + pd.Timedelta(days=1)

joined = pd.merge_asof(hourly.reset_index().sort_values("time"), daily_avail[["available_from", "prev_day_mean"]].sort_values("available_from"),
                       left_on="time", right_on="available_from", direction="backward")
joined.set_index("time").loc["2022-01-05 22:00":"2022-01-06 01:00"]

label 2022-01-05 covers: 2022-01-05 00:00:00+00:00 -> 2022-01-05 23:00:00+00:00
daily mean labelled 2022-01-05: 33942.3


,consumption_mwh,available_from,prev_day_mean
time,,,
2022-01-05 22:00:00+00:00,32604.5,2022-01-05 00:00:00+00:00,32808.137500
2022-01-05 23:00:00+00:00,30608.3,2022-01-05 00:00:00+00:00,32808.137500
2022-01-06 00:00:00+00:00,30808.0,2022-01-06 00:00:00+00:00,33942.295833
2022-01-06 01:00:00+00:00,30059.8,2022-01-06 00:00:00+00:00,33942.295833


In [23]:
# label / closed for intra-day bins: which timestamp names the bin, and which edge is included
s = pd.Series(np.arange(6), index=pd.date_range("2023-01-01", periods=6, freq="h", tz="UTC"))
pd.DataFrame({
    "default (left,left)": s.resample("3h").sum(),
    "label=right closed=right": s.resample("3h", label="right", closed="right").sum(),
})

,"default (left,left)",label=right closed=right
2023-01-01 00:00:00+00:00,3.0,0
2023-01-01 03:00:00+00:00,12.0,6
2023-01-01 06:00:00+00:00,NaN,9


## 12. Leakage checklist

Ask these before trusting any metric:

1. What is the **decision time** of each row, and what is the **horizon**?
2. Is the frame **sorted by time**, with **unique timestamps** and a **constant interval**? (Otherwise `shift(k)` is not k hours.)
3. Does every feature use only data with timestamp **≤ decision time**? Look at every `rolling`, `expanding`, `diff`, `pct_change` for a missing `shift`.
4. Any `center=True`, `bfill`, `interpolate`, `shift(-k)` that is not the target?
5. Were **exogenous variables** (weather, prices, volumes) *measured* at the target time or *forecast* at the decision time? Which forecast **vintage**?
6. Were **scalers / imputers / encoders** fit on train only (ideally inside a `Pipeline`)?
7. Are **group statistics** (means by hour, by customer, by month) computed on train only?
8. Was the split **chronological**, with a **gap ≥ h** between train and test?
9. If cross-validating: `TimeSeriesSplit`, never shuffled folds, when labels overlap or autocorrelate.
10. Is the **target** what you think it is (units, sign, horizon, aggregation window)? Check one row by hand.
11. After merges: did the row count change? Any timestamps from the right table **later than the decision time**?
12. Is the result **too good**? An R² that jumps after a "small" change is a bug until proven otherwise.